# Cat/Dog CNN Features: What Responds, and Where?

Train two ResNet-18 Cat/Dog classifiers, then see what their channels respond to
and which regions influence a prediction. The labels are **cat=0, dog=1**;
the 37 breed annotations are used for splitting, not a breed prediction head.

Display saved pictures only when their registered evidence matches the current
run; otherwise show what is missing. There is no historical substitute. These
pictures describe current model behavior, not how a filter originally learned
a pattern or a robustness guarantee.

## 1. Setup and data

Select this project's `.venv/bin/python` (CPython 3.13.15, requirements.txt,
PyTorch/MPS, tqdm). `RUN_FULL_EXPERIMENT=True` intentionally runs setup,
both training arms, eight feature families and the report. Set it to `False` for
read-only inspection; missing evidence is reported rather than substituted.

In [ ]:
import os
import sys
from pathlib import Path

configured_root = os.environ.get("OXFORD_PETS_NOTEBOOK_ROOT")
candidates = [Path(configured_root).resolve()] if configured_root else []
candidates.extend([Path.cwd().resolve(), *Path.cwd().resolve().parents])
ROOT = next(
    (
        candidate
        for candidate in candidates
        if (candidate / "configs/experiment.yaml").is_file()
        and (candidate / "src/notebook_support.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from its project checkout")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"

import torch
from IPython.display import Image, Markdown, display

from src.notebook_support import inspect_data, notebook_context, run_stage
from src.runtime_visuals import (
    display_features,
    display_report,
    display_training,
    training_observer,
)

RUN_FULL_EXPERIMENT = False
config, RUN_FULL_EXPERIMENT, device = notebook_context(full=RUN_FULL_EXPERIMENT)
print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · device: {device}")
print(f"Epochs: {config.integer('training', 'epochs')} · full execution: {RUN_FULL_EXPERIMENT}")

### Prepare the registered split

Oxford-IIIT Pet: about 7,349 photographs, 37 breeds (12 cat / 25 dog breeds).
Preserve the official test split. Hash-split trainval 80/20 within breed, then
reserve 10% of the original training pool for clean validation:
approximately **2,649 fitting / 295 validation / 736 reference / 3,669 test**.
Both models use the same IDs. Reference images select channels; validation
monitors training. Neither selects the final checkpoint.

In [ ]:
setup_result = run_stage(config, "setup", full=RUN_FULL_EXPERIMENT, device=device)
print("Dataset manifests prepared." if RUN_FULL_EXPERIMENT else "Setup skipped in safe mode.")

In [ ]:
from src.eda import generate_eda

if (config.project_path("artifacts") / "data/manifest.json").is_file():
    data_summary = inspect_data(config)
    rows = [
        ("Fitting", "training"),
        ("Validation", "validation"),
        ("Reference", "calibration"),
        ("Official test", "official_test"),
    ]
    table = "| Partition | Images |\n|---|---:|\n"
    table += "\n".join(
        f"| {label} | {data_summary.get(key, 'unavailable')} |" for label, key in rows
    )
    display(Markdown(table))
    eda = generate_eda(config)
    display(Image(filename=str(ROOT / eda["figures"]["split_and_species"]), width=1000))
else:
    display(Markdown("Data unavailable. Run the setup cell in full mode first."))

## 2. Training

Start from identical ImageNet ResNet-18 weights and a new two-class head:
the networks already know many image patterns; they are not trained from scratch.
Match fitting IDs, augmentation, sample order, optimizer updates and schedule.
Use the configured final epoch, not a validation-selected checkpoint.

Each completed epoch refreshes **training-objective loss** (the mistake penalty;
lower is better for that model's objective) and **clean validation accuracy**
(correct answers on pictures not used to fit the weights). Read the final
cat/dog recalls too: overall accuracy can hide always choosing the common class.
The dashed line gives cats and dogs equal weight. The dotted line shows what
simply guessing the more common animal would achieve—without learning the task.
Monitoring does not update the model. One point means one measured epoch.

### Standard model

Cross-entropy on actual clean augmented inputs. Training uses float32 native MPS,
micro-batch 16, two-step accumulation, AdamW and tqdm.

In [ ]:
standard_result = run_stage(
    config,
    "train",
    full=RUN_FULL_EXPERIMENT,
    device=device,
    arm="standard",
    epoch_observer=training_observer(config, compact=True) if RUN_FULL_EXPERIMENT else None,
)
display_training(config, "standard", compact=True, summary_only=RUN_FULL_EXPERIMENT)

### PGD-trained model

Train on inputs changed slightly to make classification harder. PGD takes five
small steps, keeping each pixel change within `4/255` (step `1/255`) and valid
`[0,1]` pixels. The loss is measured on these difficult inputs, so it is not the
standard model's clean loss. Validation uses clean pictures for both models.
PGD training alone does not establish robustness.

In [ ]:
adversarial_result = run_stage(
    config,
    "train",
    full=RUN_FULL_EXPERIMENT,
    device=device,
    arm="adversarial",
    epoch_observer=training_observer(config, compact=True) if RUN_FULL_EXPERIMENT else None,
)
display_training(config, "adversarial", compact=True, summary_only=RUN_FULL_EXPERIMENT)

## 3. Learned features

Use the same fixed clean test anchors: **two cats and two dogs**. Select channels
on reference images before looking at the test pictures. Each cell below computes
only its named family and necessary prerequisites, then reads verified current-run
figures. Photo panels remain ignored local artifacts.

Compact layer/activation previews show the first fixed cat and first fixed dog
for both models—not the prettiest examples. All four anchors and full figures
remain linked; use `display_features(config, section="stages", compact=False)`
(or another section) to display everything.

Read in order: layer walkthrough → learned kernels → synthetic preferences →
strong real patches → clean responses. Do not assign anatomical detector labels
from visual resemblance alone.

### 3a. Image → layers → prediction

Follow the actual 224px input through the network. A channel is a small pattern tester; its map shows where it responds, not a restored photograph. Later maps are coarser. Pooling turns the final maps into 512 summary numbers, which the classifier combines into cat/dog scores.

In [ ]:
feature_stages_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="stages"
)
display_features(config, section="stages", compact=True)

### 3b. First-layer filters

A first-layer filter is a small colored stencil that responds to patterns such as edges or color changes. Compare the starting ImageNet stencil, the fine-tuned one, and their difference. These are weights, not a heatmap for a particular pet or proof of an eye/ear/fur detector.

In [ ]:
feature_kernels_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="kernels"
)
display_features(config, section="kernels", compact=True)

### 3c. Synthetic channel preferences

Let an optimizer draw a picture that increases one channel's response. The patterns show preferences, not remembered pets or training photographs. A gray tile means this trial did not find a stronger pattern from its starting noise; it does not mean the filter is useless. Unsuccessful trials stay visible.

In [ ]:
feature_synthetic_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="synthetic"
)
display_features(config, section="synthetic", compact=True)

### 3d. Strong real-image patches

Find real reference pictures that strongly activate the selected pattern testers. The box is the region that could feed that response (its receptive field), not proof that every enclosed pixel matters or that this picture taught the filter. A late-layer box may cover the whole input. The compact view shows the first two reference-ranked channels per level and their strongest recorded image; the fuller galleries remain linked.

In [ ]:
feature_real_patches_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="real_patches"
)
display_features(config, section="real_patches", compact=True)

### 3e. Clean activation maps and sensitivity

An activation map asks 'where does this channel respond?' An input gradient asks 'which tiny pixel changes could alter that response?' These are different views, not reconstructed images. Heatmaps are display-scaled; brighter colors do not make one model better.

In [ ]:
feature_activations_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="activations"
)
display_features(config, section="activations", compact=True)

## 4. Prediction influence

Grad-CAM highlights regions linked to the animal's correct-class score
(**true-species logit**). Occlusion covers regions and measures how the correct
score's lead over the other score changes (**true-species-vs-other logit margin**).
Keep that target fixed even when the model is wrong. These methods ask different
questions; neither reveals which training picture taught a filter.

### 4a. Grad-CAM class influence

Highlight regions connected to the score for the animal's true species, even when the prediction is wrong. A cat picture always targets the cat score. Grad-CAM is coarse and a blank map means no positive map for this target under this method—not that the network has no features. It is not causal proof.

In [ ]:
feature_gradcam_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="gradcam"
)
display_features(config, section="gradcam", compact=True)

### 4b. Masking and region controls

Cover one tile at a time and watch the correct-species score minus the other score (the margin). Red/positive means covering the tile lowers that margin: it was helping the correct species. Blue/negative means covering it raises the margin. Masks are artificial inputs, and model score scales can differ.

In [ ]:
feature_occlusion_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="occlusion"
)
display_features(config, section="occlusion", compact=True)

### 4c. Randomized-weight checks and region controls

Replace the learned weights with random ones and compare Grad-CAM on the same pictures. This checks dependence on learned weights; a changed map does not prove the explanation is correct. Compare masking Grad-CAM-ranked tiles with random equal-area tiles. Flat maps can make correlation undefined. Extra response/change charts are linked as optional details.

In [ ]:
feature_diagnostics_result = run_stage(
    config, "represent", full=RUN_FULL_EXPERIMENT, device=device, section="diagnostics"
)
display_features(config, section="diagnostics", compact=True)

## 5. Current-run results

Build a feature-only report and read-only picture companion from the saved training
history and eight registered figure families. No separate attack, corruption or
confidence evaluation is needed. Incomplete evidence stays explicitly incomplete.
The companion under `reports/generated/` can be read later without training again.

You can review saved outputs without clearing them or retraining. Local photographs
and derived panels are ignored; reviewed synthetic/aggregate exports are separate
from executed notebook outputs.

In [ ]:
report_result = run_stage(config, "report", full=RUN_FULL_EXPERIMENT, device=device)
display_report(config, compact=True)

Reproduce with `.venv/bin/python src/cli.py reproduce --config configs/experiment.yaml --device mps`.
Edit valid parameters in `configs/experiment.yaml` before a fresh run; changed
identities cannot silently reuse old checkpoints. The registered run used 15 epochs;
its pure-PGD arm collapsed to the dog-majority rule, so its feature pictures are
failure diagnostics rather than evidence of model superiority or robustness.

Feature-visualization inspiration:
[Lee et al., 2009](https://ai.stanford.edu/~ang/papers/icml09-ConvolutionalDeepBeliefNetworks.pdf),
[Grad-CAM](https://arxiv.org/abs/1610.02391),
[sanity checks](https://arxiv.org/abs/1810.03292).
Our discriminative ResNet does not reproduce Lee et al.'s generative method.